# Telecom Customer Churn Prediction — Interview-Ready Version

**Goal:** Predict whether a telecom customer is likely to churn and identify the customer characteristics associated with churn.

This version keeps the original project idea and dataset, but improves:
- reproducibility and code organization
- train/test methodology
- categorical preprocessing
- prevention of SMOTE leakage during cross-validation
- model comparison using business-relevant metrics
- hyperparameter tuning
- feature importance / interpretation
- probability-based prediction
- interview-ready business conclusions


## 1. Business Problem

Telecom companies lose revenue when customers leave. A churn model can help identify customers who are more likely to leave so that retention teams can prioritize them.

**Target:** `Churn`  
- `1` = customer churned
- `0` = customer did not churn

**Important interview point:** Accuracy alone is not enough. We also examine precision, recall, F1-score, ROC-AUC and the confusion matrix because missing a real churn customer can have a business cost.


In [ ]:
# 2. Imports and reproducibility
import warnings
warnings.filterwarnings("ignore")

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier

RANDOM_STATE = 42


## 3. Load and inspect the data

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape:", df.shape)
display(df.head())
display(df.info())


In [ ]:
# Remove identifier: it is not a predictive customer-behavior feature
df = df.drop(columns=["customerID"])

# Convert TotalCharges safely
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(df.isnull().sum().sort_values(ascending=False).head(10))
print("\nTarget distribution:")
print(df["Churn"].value_counts())
print("\nTarget percentage:")
print(df["Churn"].value_counts(normalize=True).mul(100).round(2))


## 4. Exploratory Data Analysis

The purpose of EDA is not only to make plots. We want to discover **which customer groups have different churn rates** and turn those observations into business questions.


In [ ]:
# Basic numerical summary
display(df.describe(include="all").T)


In [ ]:
# Churn rate
churn_rate = df["Churn"].value_counts(normalize=True).get("Yes", 0) * 100
print(f"Overall churn rate: {churn_rate:.2f}%")


In [ ]:
# Numerical distributions
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in numeric_cols:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(data=df, x=col, hue="Churn", kde=True, element="step", ax=ax)
    ax.set_title(f"{col} distribution by churn")
    plt.show()


In [ ]:
# Churn rate by important categorical variables
eda_cols = [
    "Contract", "InternetService", "PaymentMethod",
    "TechSupport", "OnlineSecurity", "PaperlessBilling",
    "SeniorCitizen", "Partner", "Dependents"
]

for col in eda_cols:
    churn_by_group = pd.crosstab(
        df[col], df["Churn"], normalize="index"
    ).mul(100).round(2)

    if "Yes" in churn_by_group.columns:
        print(f"\n{col} — churn rate (%)")
        display(churn_by_group[["Yes"]].sort_values("Yes", ascending=False))

        plt.figure(figsize=(7, 4))
        sns.barplot(
            x=churn_by_group.index.astype(str),
            y=churn_by_group["Yes"].values
        )
        plt.title(f"Churn rate by {col}")
        plt.ylabel("Churn rate (%)")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()


### EDA questions to answer in your final presentation

Do not simply say “I created countplots.” Be ready to answer:

1. Which contract type has the highest churn rate?
2. How does churn vary with tenure?
3. Does monthly charge appear related to churn?
4. Which payment methods show higher churn?
5. Do customers without support/security services churn differently?
6. What customer segment should a retention team investigate first?

Only state an insight after calculating it from the dataset.


## 5. Prepare features and target

In [ ]:
# Convert target to 0/1
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))


## 6. Preprocessing pipeline

The original notebook used `LabelEncoder` on every categorical feature. For nominal categories, this can create an artificial numeric ordering.

Here we use `OneHotEncoder(handle_unknown="ignore")`.

The preprocessing is placed inside the model pipeline so that information from the test set is not used while fitting the model.


In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


## 7. Baseline model comparison

We compare a simple linear baseline with tree-based models.

**Important:** SMOTE is placed *inside* the cross-validation pipeline. This prevents synthetic observations from being generated using information that can cross CV fold boundaries.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []

for name, model in models.items():
    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC_AUC": scores["test_roc_auc"].mean()
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df.round(3))


## 8. Hyperparameter tuning

The original notebook selected Random Forest using default parameters. A stronger project tunes the model and chooses the final model using an appropriate validation metric.

For churn, **F1 and recall should be considered alongside ROC-AUC**, rather than selecting solely on accuracy.


In [ ]:
rf_pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)
print("\nBest CV F1:", round(search.best_score_, 3))


## 9. Final evaluation on the untouched test set

In [ ]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall   :", round(recall_score(y_test, y_pred), 3))
print("F1-score :", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 3))

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.show()


### How to explain the confusion matrix

- **True Negative:** correctly predicted a non-churning customer.
- **False Positive:** predicted churn, but the customer did not churn.
- **False Negative:** customer actually churned, but the model missed them.
- **True Positive:** correctly identified a customer who churned.

For a retention use case, false negatives are especially important because they represent churners the company failed to identify.


## 10. Model interpretation

Use permutation importance to understand which original customer features are most useful to the final model.

This is preferable to presenting encoded column names without context.


In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(8, 6))
sns.barplot(
    data=importance_df.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Top Features by Permutation Importance")
plt.tight_layout()
plt.show()


## 11. Business insights

Fill this section **after running the improved notebook**. Do not invent numbers.

A strong final presentation should contain 3–5 evidence-based findings such as:

- “Customers in ___ contract type had a churn rate of ___%.”
- “Customers with tenure below ___ months had a churn rate of ___%.”
- “The churn rate for ___ payment method was ___%.”
- “The model's recall for churn was ___, meaning it identified ___% of actual churners in the test set.”
- “The top model features were ___, ___ and ___ according to permutation importance.”

These connect machine learning to a business decision.


## 12. Customer-level prediction

Instead of returning only `0` or `1`, return a churn probability.

This gives a retention team a more useful risk score.


In [ ]:
def predict_churn(customer_data: dict):
    customer_df = pd.DataFrame([customer_data])

    probability = best_model.predict_proba(customer_df)[0, 1]
    prediction = int(probability >= 0.50)

    return {
        "prediction": "Churn" if prediction == 1 else "No Churn",
        "churn_probability": round(float(probability), 4)
    }

sample_customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 1,
    "PhoneService": "No",
    "MultipleLines": "No phone service",
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 29.85,
    "TotalCharges": 29.85
}

print(predict_churn(sample_customer))


## 13. Save the complete model pipeline

The original notebook saved the model and separate encoders. The improved approach saves the **entire preprocessing + SMOTE + model pipeline**, reducing the risk that deployment preprocessing differs from training preprocessing.


In [ ]:
with open("telecom_customer_churn_pipeline.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Saved: telecom_customer_churn_pipeline.pkl")


# Final project story for interviews

### 60-second version

> “My project is Telecom Customer Churn Prediction. The goal was to identify customers who are likely to leave a telecom service so that the company can prioritize retention efforts. I performed data cleaning and exploratory analysis, converted the target into a binary variable, and built a preprocessing pipeline for numerical and categorical features. Because churn is an imbalanced classification problem, I handled class imbalance using SMOTE inside the cross-validation pipeline to avoid data leakage. I compared Logistic Regression, Decision Tree, Random Forest and XGBoost, then tuned the final model using cross-validation. I evaluated it on an untouched test set using accuracy, precision, recall, F1-score, ROC-AUC and a confusion matrix. Finally, I used permutation importance to identify the customer features most associated with predictive performance and built a probability-based customer churn prediction function.”

### Interview topics you must be able to explain

1. What is customer churn?
2. Why is this a classification problem?
3. Why did you remove `customerID`?
4. Why was `TotalCharges` converted to numeric?
5. Why can accuracy be misleading for churn?
6. What is class imbalance?
7. What is SMOTE?
8. Why must SMOTE be performed inside CV/training folds?
9. Why did you use OneHotEncoder instead of LabelEncoder?
10. Difference between precision and recall?
11. What does the confusion matrix tell you?
12. Why did you compare multiple models?
13. Why use cross-validation?
14. What is hyperparameter tuning?
15. Why evaluate once on an untouched test set?
16. What does ROC-AUC mean?
17. How would the telecom company use the prediction?
18. What are the limitations of your model?


# Portfolio upgrade checklist

- [ ] Run every cell from top to bottom in a fresh kernel.
- [ ] Record the final test metrics.
- [ ] Record the 3–5 strongest EDA findings.
- [ ] Record the top model features.
- [ ] Add a README with the business problem, approach, metrics and findings.
- [ ] Create a Power BI/Tableau dashboard using the same dataset.
- [ ] Optionally deploy the saved pipeline with Streamlit.
- [ ] Put the project on GitHub with a clean folder structure.
- [ ] Practice the 18 interview questions above.
